# Tanzania VACS 2009 — PUD exploration (Male + Female)

Separate public files under **`data/raw/Tanzania 2009 Stata/`**:
- **`TANZANIA_VACS_2009_male_PUD_UR.dta`**
- **`TANZANIA_VACS_2009_Female_PUD_UR.dta`**

Split-sample EAs (males and females interviewed in different enumeration areas). **ID prefixes differ:** male `MM…`, female `MF…` (mainland) and `ZF…` (Zanzibar).

**Encoding:** Stata strings may require **`latin1`** for `pyreadstat.read_dta` (UTF-8 can fail).

**Flow:** (1) Load both → (2) §2 column list & quick EDA → (3) §3 further EDA, **checklist** (`utils.checklist`, **IDZ** first for `type_and_width`), slot summaries → (4) §4 harmonized codebook (copyable TSV). After editing `utils/`, **restart kernel**.

**Harmonized Excel:** aim for **one row per country / wave**; put male vs female variable names in **separate columns** and spell out differences only in **notes** (§4 TSV follows that layout).

Documentation PDFs in the same folder include **`TANZANIA_VACS_2009_DataUserGuide.pdf`** (design: clustering, stratification, weights).


In [ ]:
from pathlib import Path
import sys

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

COUNTRY_DIR = ROOT / "data" / "raw" / "Tanzania 2009 Stata"
MALE_DTA = COUNTRY_DIR / "TANZANIA_VACS_2009_male_PUD_UR.dta"
FEMALE_DTA = COUNTRY_DIR / "TANZANIA_VACS_2009_Female_PUD_UR.dta"
ENC = "latin1"

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


## 1. Load data

Use **`encoding=ENC`** (`latin1`). `read_dta` returns `df` and `meta`.


In [ ]:
for p in (MALE_DTA, FEMALE_DTA):
    if not p.is_file():
        raise FileNotFoundError(f"Expected:\n  {p}")

df_m, meta_m = pyreadstat.read_dta(MALE_DTA, encoding=ENC)
df_f, meta_f = pyreadstat.read_dta(FEMALE_DTA, encoding=ENC)

print(f"Male:   {MALE_DTA.name}  →  {df_m.shape[0]:,} × {df_m.shape[1]:,}")
print(f"Female: {FEMALE_DTA.name}  →  {df_f.shape[0]:,} × {df_f.shape[1]:,}")
print(f"Male duplicated rows (all columns):   {int(df_m.duplicated().sum())}")
print(f"Female duplicated rows (all columns): {int(df_f.duplicated().sum())}")

print("\nMale IDZ prefix (first 2 chars):")
display(df_m["IDZ"].astype(str).str[:2].value_counts())
print("Female IDZ prefix:")
display(df_f["IDZ"].astype(str).str[:2].value_counts())


## 2. Column list & quick EDA

Stata labels, dtypes, missingness (first 40 columns + top missing) **for each file**.


In [ ]:
def var_table_for(df: pd.DataFrame, meta, title: str):
    name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}
    vt = pd.DataFrame({
        "column": df.columns,
        "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
        "dtype": df.dtypes.astype(str).values,
        "missing_n": df.isna().sum().values,
        "missing_pct": (100 * df.isna().mean()).round(2),
    })
    print(title)
    print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
    display(vt.head(40))
    display(vt.sort_values("missing_pct", ascending=False).head(15).reset_index(drop=True))


var_table_for(df_m, meta_m, "— Male PUD —")
var_table_for(df_f, meta_f, "— Female PUD —")


## 3. Further EDA and exploration

### Raw row samples

**Core** columns for IDs, geography, cluster, weights, dates.


In [ ]:
pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 70)

_core = [
    c
    for c in [
        "IDZ",
        "CLUSTERz",
        "Urbanrural",
        "stratum",
        "postweight",
        "IQC",
        "TLQC",
        "V1Date",
        "V2Date",
        "V3Date",
        "hhyy",
        "teamz",
        "EA_OK",
        "EA_OE",
    ]
    if c in df_m.columns or c in df_f.columns
]

for label, df in [("MALE", df_m), ("FEMALE", df_f)]:
    cols = [c for c in _core if c in df.columns]
    sub = df[cols]
    print("\n" + "=" * 70)
    print(label, "head(6)")
    display(sub.head(6))
    print(label, "sample(5, random_state=0)")
    display(sub.sample(5, random_state=0))


### Harmonized geography / ID checklist (`utils.checklist`)

**Tanzania 2009 — Male and Female PUDs** (separate `df_m` / `df_f`). **`IDZ`** is listed first so **`type_and_width`** and **`suggested_layout`** are easy to find.

**Admin template:** **`IQC`** → Admin **1** (male file only); **`TLQC`** → **~1.5**; **`CLUSTERz`** → **Admin 2 (EA)** + cluster for `svy`. Edit **`CANDIDATES`** if needed.

See **`skills/memory.md`** (PI width = usual character count).


In [ ]:
from utils.checklist import build_checklist_df, checklist_to_tsv

CANDIDATES = [
    ("Respondent / record ID (IDZ)", ["IDZ"]),
    ("Admin 1 (IQC — present male PUD only)", ["IQC"]),
    ("Admin ~1.5 (TLQC)", ["TLQC"]),
    ("Urban / rural", ["Urbanrural"]),
    ("Admin 2 — EA (CLUSTERz)", ["CLUSTERz"]),
    ("Stratum", ["stratum"]),
    ("Weight", ["postweight"]),
    ("Visit dates (text)", ["V1Date", "V2Date", "V3Date"]),
]

for label, df, meta in [("MALE", df_m, meta_m), ("FEMALE", df_f, meta_f)]:
    print("\n" + "=" * 72)
    print(f"CHECKLIST — {label}")
    print("=" * 72)
    _labels = meta.column_names_to_labels or {}
    checklist_df = build_checklist_df(df, CANDIDATES, column_labels=_labels)
    with pd.option_context("display.max_colwidth", 100, "display.width", 220):
        display(checklist_df)
    print(f"\n--- TSV ({label}) — copy for Excel ---\n")
    print(checklist_to_tsv(checklist_df))


### Slot summaries (ID / geo / cluster / weight / date)

Same compact width line as Tanzania 2024 v1: **chars** for strings, **integer-code digit counts** for whole-number numerics, optional **style** template, **M/F token** heuristic on strings.

Runs **separately** on male and female extracts (variable sets differ slightly: e.g. **`IQC` is male-only**).

For **`V1Date` / `V2Date` / `V3Date`**, the harmonized codebook documents the literal pattern as **`DD/MM/YYYY`** (see §4 TSV); carry the same wording into Excel **notes** when you paste.


In [ ]:
_WORD_SEX = re.compile(r"\b(male|females?|female)\b", re.IGNORECASE)
# Underscore-delimited segments like Female_… in long IDs (not single-letter TLQC/IQC codes "M", "F")
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")


def _abstract_digit_pattern(val: str) -> str:
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(which: str, title: str, cols: list, df: pd.DataFrame, meta, note_extra: str = ""):
    L = meta.column_names_to_labels or {}
    print("\n" + "=" * 60)
    print(which, "-", title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (L.get(c) or "")[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


cols_id = ["IDZ"]
cols_hh = ["CLUSTERz"]  # no separate hh in PUD; cluster only
cols_geo1 = ["IQC", "TLQC"]  # IQC female: missing column
cols_geo2 = ["Urbanrural"]
cols_clust = ["CLUSTERz", "stratum", "postweight"]
cols_date = ["V1Date", "V2Date", "V3Date"]

for label, df, meta in [("MALE", df_m, meta_m), ("FEMALE", df_f, meta_f)]:
    uid = df["IDZ"].nunique()
    slot_summary(
        label,
        "1. Respondent ID",
        cols_id,
        df,
        meta,
        f"unique IDZ: {uid} / {len(df)} rows",
    )
    slot_summary(label, "2. Household / cluster context", cols_hh, df, meta, "no separate household ID column in PUD")
    slot_summary(label, "3. Geo codes (IQC / TLQC)", cols_geo1, df, meta, "")
    slot_summary(label, "4. Urban / rural (Swahili labels)", cols_geo2, df, meta, "")
    slot_summary(label, "5. Cluster + stratum + weight", cols_clust, df, meta, "")
    slot_summary(label, "6. Interview / visit dates", cols_date, df, meta, "")

print("\n--- Duplicate IDZ within file (female, first 10 keys) ---")
d = df_f[df_f["IDZ"].duplicated(keep=False)].sort_values("IDZ")
if len(d):
    display(d[["IDZ", "CLUSTERz", "Urbanrural", "V1Date", "postweight"]].head(10))
else:
    print("none")


## 4. Harmonized codebook slots (Tanzania 2009)

**Excel convention:** **One row per country / survey wave.** Use **`variable_male`** and **`variable_female`** for the Stata names in each PUD—the same slot is **not** duplicated on two rows. Put sex-specific differences, missing columns, and **stable shapes** (date order, code widths) in **`notes`** / **`type_and_width`**.

**Data sources:** `TANZANIA_VACS_2009_male_PUD_UR.dta` and `TANZANIA_VACS_2009_Female_PUD_UR.dta` in `data/raw/Tanzania 2009 Stata/`. Use **`latin1`** with `pyreadstat`. See **`TANZANIA_VACS_2009_DataUserGuide.pdf`** for design.

When a field has a fixed template, record it explicitly (e.g. visit dates as **`DD/MM/YYYY`**—consistent with values like **`19/11/2009`** in the extract).

Plain TSV (paste into Excel; add survey-year / country key in your sheet as you do elsewhere):

```
slot	variable_male	variable_female	type_and_width	notes
Respondent ID	IDZ	IDZ	str; prefixes MM (M) vs MF/ZF (F); char widths ~6–7 (M), ~2–7 (F)	Split-sample EAs; no ID overlap across files; **exact duplicate rows** in each file (~242 M, ~207 F)—dedupe if you need one row per IDZ
Household ID	—	—	—	No `hh` column; **CLUSTERz** is EA/cluster, not household; one selected adolescent per sampled HH (Kish)
Geo level 1	IQC	TLQC	str; short letter codes	Female file **lacks IQC**—first geo layer is **TLQC** only; male uses **IQC** then TLQC as level 2
Geo level 2	TLQC	Urbanrural	str; TLQC 0–5 chars; Urbanrural Kiswahili (e.g. Mjini, Vijijini)	Male: TLQC under IQC; female: urban/rural after TLQC
Geo level 3	—	—	—	Not separately coded in PUD beyond the rows above
Cluster	CLUSTERz	CLUSTERz	numeric cluster / EA codes 1–200 (integer string 1–3 digits)	Same variable both files; **not** hyphenated like TZ 2024
Stratum	stratum	stratum	1–2 (integer codes)	Design stratum
Weight	postweight	postweight	float (numeric as string width varies)	Analysis weight; see notebook min/max
Interview date	V1Date, V2Date, V3Date	V1Date, V2Date, V3Date	str; **DD/MM/YYYY** (e.g. 19/11/2009; 10 chars when complete)	Empty string when visit absent; V3 more often populated on female file
```

**M/F heuristic (slot summaries):** Single-letter codes **`M`** and **`F`** in **TLQC / IQC** are geographic / survey codes, **not** sex markers. The notebook now flags only **`male` / `female` words** or underscore-delimited **`Male` / `Female`** segments (e.g. long IDs)—validated: TLQC no longer shows a false sex flag.
